In [1]:
from csrio_image2biomass.configs.settings import AUGUMENTED_DATA_DIR
import polars as pl
train = pl.read_csv(AUGUMENTED_DATA_DIR / "train.csv")
test = pl.read_csv(AUGUMENTED_DATA_DIR / "test.csv")
train.sort("image_path").head()

image_path,Dry_Clover_g,Dry_Dead_g,Dry_Green_g,Dry_Total_g,GDM_g
str,f64,f64,f64,f64,f64
"""train/ID1011485656.jpg""",0.0,31.9984,16.2751,48.2735,16.275
"""train/ID1011485656_hflip.jpg""",0.0,31.9984,16.2751,48.2735,16.275
"""train/ID1011485656_hvflip.jpg""",0.0,31.9984,16.2751,48.2735,16.275
"""train/ID1011485656_vflip.jpg""",0.0,31.9984,16.2751,48.2735,16.275
"""train/ID1012260530.jpg""",0.0,0.0,7.6,7.6,7.6


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms, models
from PIL import Image
import numpy as np
import os
from typing import Dict, Any, Tuple

class BiomassDataset(Dataset):
    def __init__(self, dataframe: pl.DataFrame, img_dir: str):
        self.dataframe = dataframe
        self.img_dir = img_dir
        self.transform = transforms.Compose([
            transforms.Resize((224, 448)),
            transforms.ToTensor(),
        ])

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx) -> Tuple[torch.Tensor, torch.Tensor]:
        item: Dict[str, Any] = self.dataframe.row(idx, named=True)
        img_name = os.path.join(self.img_dir, item['image_path'])
        image = Image.open(img_name).convert('RGB')
        labels = np.array([item['Dry_Clover_g'], item['Dry_Dead_g'], item['Dry_Green_g']], dtype=np.float32)
        image = self.transform(image)
        labels = torch.tensor(labels, dtype=torch.float32)

        return image, labels
    
train_dataset = BiomassDataset(dataframe=train, img_dir=str(AUGUMENTED_DATA_DIR))
test_dataset = BiomassDataset(dataframe=test, img_dir=str(AUGUMENTED_DATA_DIR))
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=12)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=12)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

for batch in train_loader:
    images, labels = batch
    print(f"Image batch shape: {images.size()}")
    print(f"Label batch shape: {labels.size()}")
    break


Using device: cuda
Image batch shape: torch.Size([32, 3, 224, 448])
Label batch shape: torch.Size([32, 5])


In [8]:
model = models.resnet18()
num_ftrs = model.fc.in_features
model.fc = nn.Linear(num_ftrs, 5)
model = model.to(device)
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)
num_epochs = 20
for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)

    epoch_loss = running_loss / len(train_loader.dataset)
    print(f"Epoch {epoch+1}/{num_epochs}, Loss: {epoch_loss:.4f}")

Epoch 1/20, Loss: 735.4660
Epoch 2/20, Loss: 371.8071
Epoch 3/20, Loss: 321.3328
Epoch 4/20, Loss: 313.7185
Epoch 5/20, Loss: 290.1732
Epoch 6/20, Loss: 306.0412
Epoch 7/20, Loss: 265.0925
Epoch 8/20, Loss: 246.0549
Epoch 9/20, Loss: 232.9041
Epoch 10/20, Loss: 233.4384
Epoch 11/20, Loss: 208.9935
Epoch 12/20, Loss: 194.2094
Epoch 13/20, Loss: 189.0429
Epoch 14/20, Loss: 157.5104
Epoch 15/20, Loss: 151.0486
Epoch 16/20, Loss: 135.9037
Epoch 17/20, Loss: 123.1041
Epoch 18/20, Loss: 128.5497
Epoch 19/20, Loss: 124.1056
Epoch 20/20, Loss: 122.4409


In [9]:
outputs = []
model.eval()
with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        preds = model(images)
        preds = preds.cpu()
        outputs.append(preds)

outputs = torch.cat(outputs, dim=0).numpy()
outputs

array([[ 5.4367146, 27.411936 , 19.275906 , 52.609795 , 24.619558 ]],
      dtype=float32)

In [10]:
output_df = pl.DataFrame(outputs, schema=["Dry_Clover_g", "Dry_Dead_g", "Dry_Green_g", "Dry_Total_g", "GDM_g"])
output_df

Dry_Clover_g,Dry_Dead_g,Dry_Green_g,Dry_Total_g,GDM_g
f32,f32,f32,f32,f32
5.436715,27.411936,19.275906,52.609795,24.619558


In [11]:
test_df = test.select("image_path").hstack(output_df).unpivot(on=["Dry_Clover_g", "Dry_Dead_g", "Dry_Green_g", "Dry_Total_g", "GDM_g"], index="image_path", value_name="target")
test_df

image_path,variable,target
str,str,f32
"""test/ID1001187975.jpg""","""Dry_Clover_g""",5.436715
"""test/ID1001187975.jpg""","""Dry_Dead_g""",27.411936
"""test/ID1001187975.jpg""","""Dry_Green_g""",19.275906
"""test/ID1001187975.jpg""","""Dry_Total_g""",52.609795
"""test/ID1001187975.jpg""","""GDM_g""",24.619558


In [ ]:
# concat image_path and variable columns
submission = test_df.with_columns(
    pl.concat_str([pl.col("image_path").str.split('/').list.get(-1).str.split('.jpg').list.get(0), pl.lit("_"), pl.col("variable")]).alias("sample_id")
).select(["sample_id", "target"])
print(submission)

shape: (5, 2)
┌───────────────────────────┬───────────┐
│ sample_id                 ┆ target    │
│ ---                       ┆ ---       │
│ str                       ┆ f32       │
╞═══════════════════════════╪═══════════╡
│ ID1001187975_Dry_Clover_g ┆ 1.537125  │
│ ID1001187975_Dry_Dead_g   ┆ 5.251428  │
│ ID1001187975_Dry_Green_g  ┆ 5.495736  │
│ ID1001187975_Dry_Total_g  ┆ 11.994755 │
│ ID1001187975_GDM_g        ┆ 6.773596  │
└───────────────────────────┴───────────┘


In [ ]:
submission.write_csv("submission.csv")

In [ ]:
# save model
torch.save(model.state_dict(), "biomass_model.pth")

In [7]:
train.filter(pl.col("image_path") == "train/ID196516535_hflip.jpg")

image_path,Dry_Clover_g,Dry_Dead_g,Dry_Green_g,Dry_Total_g,GDM_g
str,f64,f64,f64,f64,f64
"""train/ID196516535_hflip.jpg""",0.5231,24.8461,9.0231,34.3923,9.5462
